In [1]:
# Wyniki będą podobne jak dla RGB
import numpy as np
import cv2
import cv2.data
import matplotlib.pyplot as plt
import os

In [2]:
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

image_path = "../../../Dane/Sample/zd"
files = [f for f in os.listdir(image_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG i PNG i WEBP

images = []

for file in files:
    img = cv2.imread(str(os.path.join(image_path, file)))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append((img, file))

In [3]:
# Obliczenie kontekstu otoczenia twarzy
def get_face_context(img, face_coordinates):
    # 1. Obliczenie proporcji twarzy w stosunku do całego obrazu
    # 2. Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    # 3. Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    # 4. Zwrócenie współrzędnych z funkcji
    x, y, w, h = face_coordinates
    img_h, img_w, _ = img.shape

    # Obliczenie proporcji twarzy w stosunku do całego obrazu
    face_proportion = (w * h) / (img_h * img_w)

    # Uznajemy, że bezpośrednie otoczenie twarzy to 40% wysokości i szerokości twarzy
    # [PARAMETR]
    context_proportion = 0.40

    # Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    rectangle_point = (x - (context_proportion * w), y - (context_proportion * h))
    rectangle_width = w + 2 * (context_proportion * w)
    rectangle_height = h + 2 * (context_proportion * h)

    # Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    rectangle_point_far = (x - (2 * context_proportion * w), y - (2 * context_proportion * h))
    rectangle_width_far = w + 4 * (context_proportion * w)
    rectangle_height_far = h + 4 * (context_proportion * h)

    # Sprawdzenie, czy punkty nie wychodzą poza obraz
    if rectangle_point[0] < 0:
        rectangle_point = (0, rectangle_point[1])
    if rectangle_point[1] < 0:
        rectangle_point = (rectangle_point[0], 0)
    if rectangle_point[0] + rectangle_width > img_w:
        rectangle_width = img_w - rectangle_point[0]
    if rectangle_point[1] + rectangle_height > img_h:
        rectangle_height = img_h - rectangle_point[1]

    if rectangle_point_far[0] < 0:
        rectangle_point_far = (0, rectangle_point_far[1])
    if rectangle_point_far[1] < 0:
        rectangle_point_far = (rectangle_point_far[0], 0)
    if rectangle_point_far[0] + rectangle_width_far > img_w:
        rectangle_width_far = img_w - rectangle_point_far[0]
    if rectangle_point_far[1] + rectangle_height_far > img_h:
        rectangle_height_far = img_h - rectangle_point_far[1]

    # Zwrócenie wartości
    return {
        'face_proportion': face_proportion,
        'rectangle_point': rectangle_point,
        'rectangle_width': rectangle_width,
        'rectangle_height': rectangle_height,
        'rectangle_point_far': rectangle_point_far,
        'rectangle_width_far': rectangle_width_far,
        'rectangle_height_far': rectangle_height_far
    }

In [4]:
# Utworzenie pojedynczego histogramu
def analyze_distribution(img):
    img_data = img.flatten()

    # Utworzenie histogramu na podstawie intensywności krawędzi, podzielonego na 20 przedziałów
    # [PARAMETR]
    hist, bins = np.histogram(img_data, bins=10)

    return hist, bins

# Utworzenie "średniego" histogramu
def average_hist(histograms):
    hist_array = np.array(histograms)

    average_hist = np.mean(hist_array, axis=0)

    return average_hist

In [5]:
# Utworzenie histogramu dla pojedynczej twarzy
# W HSV
img, file = images[0]
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(hsv)

histH, binsH = analyze_distribution(h)
histS, binsS = analyze_distribution(s)
histV, binsV = analyze_distribution(v)

# plt.figure(figsize=(10, 6))
# plt.bar(binsH[:-1], histH, width=(binsH[1] - binsH[0]), color='b', alpha=0.7)
# plt.xlabel("Wartość składowej H")
# plt.ylabel("Częstotliwość występowania")
# plt.title("Histogram składowej H w przestrzeni HSV")
# plt.show()

# plt.figure(figsize=(10, 6))
# plt.bar(binsS[:-1], histS, width=(binsS[1] - binsS[0]), color='b', alpha=0.7)
# plt.xlabel("Wartość składowej S")
# plt.ylabel("Częstotliwość występowania")
# plt.title("Histogram składowej S w przestrzeni HSV")
# plt.show()

plt.figure(figsize=(10, 6))
plt.bar(binsV[:-1], histV, width=(binsV[1] - binsV[0]), color='b', alpha=0.7)
plt.xlabel("Wartość składowej V")
plt.ylabel("Częstotliwość występowania")
plt.title("Histogram składowej V w przestrzeni HSV")
plt.show()

In [6]:
# Utworzenie "średniego" histogramu dla wszystkich autentycznych twarzy
# Wczytanie wszystkich autentycznych twarzy
#image_path = "../../../Dane/Sample/Default"
image_path = "../../../Dane/Humans"
files = [f for f in os.listdir(image_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".jpeg")] # Tylko pliki JPG i PNG i JPEG

histograms_s = []
histograms_v = []

# Wybieramy tylko 500 plików (zbyt dużo danych)
files = files[:500]

for file in files:
    img = cv2.cvtColor(cv2.imread(str(os.path.join(image_path, file))), cv2.COLOR_BGR2RGB)

    faces = face_classifier.detectMultiScale(img, 1.3, 5)
    if faces is None or len(faces) == 0:
        continue
    face = faces[0]
    face_context = get_face_context(img, face)

    img_copy = img.copy()
    img_copy = cv2.cvtColor(img_copy, cv2.COLOR_RGB2HSV)

    x, y, w, h = face

    rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
    rectangle_width = int(face_context['rectangle_width'])
    rectangle_height = int(face_context['rectangle_height'])
    
    rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
    rectangle_width_far = int(face_context['rectangle_width_far'])
    rectangle_height_far = int(face_context['rectangle_height_far'])

    context_area = img_copy[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
    context_area_far = img_copy[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]

    h, s, v = cv2.split(context_area)
    hist_v, bins_v = analyze_distribution(v)
    hist_s, bins_s = analyze_distribution(s)
    histograms_v.append(hist_v)
    histograms_s.append(hist_s)
    
    h, s, v = cv2.split(context_area_far)
    hist_v, bins_v = analyze_distribution(v)
    hist_s, bins_s = analyze_distribution(s)
    histograms_v.append(hist_v)
    histograms_s.append(hist_s)

average_histogram_v = average_hist(histograms_v)
average_histogram_s = average_hist(histograms_s)

plt.figure(figsize=(10, 6))
plt.bar(bins_s[:-1], average_histogram_s, width=(bins_s[1] - bins_s[0]), color='b', alpha=0.7)
plt.xlabel("Wartość składowej S")
plt.ylabel("Częstotliwość występowania")
plt.title("Średni histogram składowej S w przestrzeni HSV")
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(bins_v[:-1], average_histogram_v, width=(bins_v[1] - bins_v[0]), color='b', alpha=0.7)
plt.xlabel("Wartość składowej V")
plt.ylabel("Częstotliwość występowania")
plt.title("Średni histogram składowej V w przestrzeni HSV")
plt.show()

In [ ]:
# 1. Obliczenie średniej i odchylenia standardowego dla każdego przedziału histogramu
# 2. Obliczenie z-score dla nowego histogramu i porównanie z wartościami z poprzedniego punktu
# bins='auto'? -> tworzy za dużo binów
def calculate_mean_and_sd(histograms):
    hist_array = np.array(histograms)
    mean_hist = np.mean(hist_array, axis=0)
    std_hist = np.std(hist_array, axis=0)
    return mean_hist, std_hist

def compare_histograms(new_histogram, mean_hist, std_hist):
    z_scores = (new_histogram - mean_hist) / std_hist
    return z_scores


mean_v, std_v = calculate_mean_and_sd(histograms_v)
mean_s, std_s = calculate_mean_and_sd(histograms_s)

print("Średnia składowej S:", mean_s)
print("Odchylenie standardowe składowej S:", std_s)
print("Średnia składowej V:", mean_v)
print("Odchylenie standardowe składowej V:", std_v)

In [ ]:
# Wczytanie przykładowego zdjęcia do porównania
image_path = "../../../Dane/Sample/zd/7.webp"
img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

# Wycięcie obszaru wokół twarzy
faces = face_classifier.detectMultiScale(img, 1.3, 5)
face = faces[0]
face_context = get_face_context(img, face)
img_copy = img.copy()

rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
rectangle_width = int(face_context['rectangle_width'])
rectangle_height = int(face_context['rectangle_height'])

rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
rectangle_width_far = int(face_context['rectangle_width_far'])
rectangle_height_far = int(face_context['rectangle_height_far'])

context_area = img_copy[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
context_area_far = img_copy[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]

hsv = cv2.cvtColor(context_area, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(hsv)
hist_s, bins_s = analyze_distribution(s)
hist_v, bins_v = analyze_distribution(v)

hsv_far = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2HSV)
h, s, v = cv2.split(hsv_far)
hist_s_far, bins_s_far = analyze_distribution(s)
hist_v_far, bins_v_far = analyze_distribution(v)

In [ ]:
# Porównanie histogramów
z_scores_s = compare_histograms(hist_s, mean_s, std_s)
z_scores_v = compare_histograms(hist_v, mean_v, std_v)
print("Z-score dla składowej S:", z_scores_s)
print("Z-score dla składowej V:", z_scores_v)

z_scores_s_far = compare_histograms(hist_s_far, mean_s, std_s)
z_scores_v_far = compare_histograms(hist_v_far, mean_v, std_v)
print("Z-score dla składowej S (dalsze otoczenie):", z_scores_s_far)
print("Z-score dla składowej V (dalsze otoczenie):", z_scores_v_far)